In [1]:
import os
import json
import pickle
import argparse
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import tgt
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [2]:
sampa_to_api_single = {
    # vowels
    'a': 'a',      # (ignore 'ɑ' duplicate → keep simple)
    'e': 'e',
    'i': 'i',
    'o': 'o',
    'u': 'u',
    'y': 'y',

    '2': 'ø',
    '9': '9',      # œ
    '@': 'ə',
    'E': 'ɛ',
    'O': 'ɔ',

    # nasal vowels
    'a~': '@',     # ɑ̃
    'e~': '5',     # ɛ̃
    '9~': '1',     # œ̃ / ɛ̃
    'o~': '§',     # ɔ̃

    # consonants
    'b': 'b',
    'd': 'd',
    'f': 'f',
    'g': 'ɡ',
    'k': 'k',
    'l': 'l',
    'm': 'm',
    'n': 'n',
    'n=':'n',
    'p': 'p',
    't': 't',
    'v': 'v',
    'w': 'w',
    'z': 'z',
    'j': 'j',

    # special consonants
    'R': 'ʁ',
    'N': 'ŋ',
    'H': 'ɥ',

    # palatal / special
    'J': 'ɲ',      # prefer ɲ over ɟ (French consistent)
    'S': 'ʃ',      # prefer fricative over affricate
    'Z': 'ʒ',
    'Z=': 'ʒ',

    # affricates (if explicitly needed)
    's': 's',

    # special cases from your table
    'm=': 'm',

    # silence / noise
    '_': '_',
    'spn': 'spn',
    'unk': 'spn',
    "%":'spn',
    "?":"spn",
    "0":"spn"
}
hyp_to_ref = {
    **sampa_to_api_single,
    '@': '@',    # override: vocab '@' is the nasal vowel, not schwa
    'ts': 's',    # ADD — model sometimes outputs affricate, collapse to /s/
    'tʃ': 'ʃ'
}
def normalise_hyp_token(tok: str) -> str:
    """Convert a raw tokenizer token to the REF label space."""
    return hyp_to_ref.get(tok, tok)   # passthrough if not in map
def map_ref_to_api_single(ph, mapping):
    if ph in mapping:
        return mapping[ph]
    if ph is not None and ph != "":
        return ph
    return None



In [3]:
with open("vocab_w2vCTC.json") as f:
    vocab = json.load(f)
print("Tokens:", sorted(vocab.keys()))
print("Number of tokens:", len(vocab))

Tokens: ['1', '5', '9', '@', '[PAD]', '[UNK]', 'a', 'b', 'd', 'dʒ', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'ts', 'tʃ', 'u', 'v', 'w', 'y', 'z', '§', 'ø', 'ŋ', 'ɔ', 'ə', 'ɛ', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Number of tokens: 40


In [4]:
def read_textgrid(tg_path, tier_name="phone"):
    """
    Returns list of {'phoneme': str, 'start': float, 'end': float}.
    Skips silence intervals (empty label, SIL, sil, sp).
    """
    tg        = tgt.io.read_textgrid(tg_path)
    tier      = tg.get_tier_by_name(tier_name)
    silence   = {"", "SIL", "sil", "spn", "SP", "<SIL>", "_","0","fe~","sjo~","Ra~"}
    intervals = []
    for iv in tier.intervals:
        label = iv.text.strip()
        phoneme=map_ref_to_api_single(label,sampa_to_api_single)
        if phoneme in silence:
            continue
        intervals.append({
            "phoneme": phoneme,
            "start":   round(iv.start_time, 6),
            "end":     round(iv.end_time,   6),
        })
        if phoneme==None:
            print(phoneme)
    return intervals

In [5]:
import Levenshtein

def align_sequences(ref, hyp):
    alignment = []
    ops = Levenshtein.editops(ref, hyp)

    ref_idx = hyp_idx = 0
    op_idx = 0

    while ref_idx < len(ref) or hyp_idx < len(hyp):

        if op_idx < len(ops):
            op_type, src_pos, dest_pos = ops[op_idx]

            if op_type == "delete" and src_pos == ref_idx:
                alignment.append((ref_idx, None))
                ref_idx += 1
                op_idx += 1
                continue

            elif op_type == "insert" and dest_pos == hyp_idx:
                alignment.append((None, hyp_idx))
                hyp_idx += 1
                op_idx += 1
                continue

            elif op_type == "replace" and \
                 src_pos == ref_idx and \
                 dest_pos == hyp_idx:
                alignment.append((ref_idx, hyp_idx))
                ref_idx += 1
                hyp_idx += 1
                op_idx += 1
                continue

        # equal case
        if ref_idx < len(ref) and hyp_idx < len(hyp):
            alignment.append((ref_idx, hyp_idx))
            ref_idx += 1
            hyp_idx += 1
        elif ref_idx < len(ref):
            alignment.append((ref_idx, None))
            ref_idx += 1
        elif hyp_idx < len(hyp):
            alignment.append((None, hyp_idx))
            hyp_idx += 1

    return alignment

In [43]:
def match_alignments_lev(ref_alignments, hyp_alignments, ref_seq, hyp_seq,
                         max_time_diff=None):  # None = no filtering by default

    alignment = align_sequences(ref_seq, hyp_seq)#align_sequences_banded(ref_seq, hyp_seq, band=band)#

    start_errors = []
    end_errors = []
    duration_errors = []
    mid_errors = []
    matched_pairs = []
    filtered_count = 0

    for ref_idx, hyp_idx in alignment:

        if ref_idx is None or hyp_idx is None:
            continue

        if ref_seq[ref_idx] != hyp_seq[hyp_idx]:
            continue

        r_start = ref_alignments[ref_idx]["start"]
        r_end   = ref_alignments[ref_idx]["end"]
        h_start = hyp_alignments[hyp_idx]["start"]
        h_end   = hyp_alignments[hyp_idx]["end"]

        mid_ref  = (r_start + r_end) / 2
        mid_pred = (h_start + h_end) / 2
        mid_error = abs(mid_ref - mid_pred)

        if max_time_diff is not None and mid_error > max_time_diff:
            filtered_count += 1
            continue

        start_errors.append(abs(r_start - h_start))
        end_errors.append(abs(r_end - h_end))
        duration_errors.append(abs((r_end - r_start) - (h_end - h_start)))
        mid_errors.append(mid_error)
        matched_pairs.append((ref_idx, hyp_idx))

    return start_errors, end_errors, duration_errors, mid_errors, matched_pairs, filtered_count

def get_phoneme_alignments(model, processor, audio_path):
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio)
    chunks = vad_chunk_with_timestamps(wav, max_pause_duration=1.5)
    device = next(model.parameters()).device
    blank_id = model.config.pad_token_id

    all_alignments = []
    full_phoneme_parts = []

    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]

        inputs = processor(
            chunk_tensor.numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)[0]
        
        decoded = processor.batch_decode(predicted_ids.unsqueeze(0))[0]
        full_phoneme_parts.append(decoded.strip())

        num_frames     = logits.shape[1]
        chunk_duration = (end_sample - start_sample) / 16000
        frame_duration = chunk_duration / num_frames

        # ✅ Reset per chunk
        prev_id = None
        current_alignment = None

        for frame_idx, token_id in enumerate(predicted_ids.tolist()):
    
            if token_id == blank_id:
                prev_id = token_id
                continue
            
            phoneme = processor.decode([token_id])
            
            # Skip empty / space / word separator
            if phoneme in ["", " ", "|"]:
                prev_id = token_id
                continue
            

            
            if token_id != prev_id:
                if current_alignment is not None:
                    current_alignment["end"] = (
                        start_sample / 16000 + frame_idx * frame_duration
                    )
            
                start_time = start_sample / 16000 + frame_idx * frame_duration
                current_alignment = {
                    "phoneme": phoneme,
                    "start": start_time,
                    "end": None
                }
                all_alignments.append(current_alignment)
            
            prev_id = token_id


        # close last phoneme of this chunk using frame-based end
        if current_alignment is not None and current_alignment["end"] is None:
            current_alignment["end"] = (
                start_sample / 16000 + num_frames * frame_duration
            )
    full_phoneme_string = " ".join(full_phoneme_parts)
    return full_phoneme_string.strip(), all_alignments
def compute_best_shift(ref_intervals, hyp_intervals, ref_seq, hyp_seq):
    """
    Find the time shift that minimizes median error between matched pairs.
    Tests shifts from -5s to +5s in 20ms steps.
    """
    from Levenshtein import editops
    
    alignment = align_sequences(ref_seq, hyp_seq)
    matched = [(ref_idx, hyp_idx) for ref_idx, hyp_idx in alignment
               if ref_idx is not None and hyp_idx is not None
               and ref_seq[ref_idx] == hyp_seq[hyp_idx]]
    
    if not matched:
        return 0.0
    
    ref_mids = np.array([(ref_intervals[r]["start"] + ref_intervals[r]["end"]) / 2 
                          for r, h in matched])
    hyp_mids = np.array([(hyp_intervals[h]["start"] + hyp_intervals[h]["end"]) / 2 
                          for r, h in matched])
    
    # Best shift = median of (ref - hyp) differences
    best_shift = np.median(ref_mids - hyp_mids)
    return best_shift
def extract_phones_from_textgrid(tg_path, remove_silence=True,t=""):
    """
    Extract phoneme sequence and timestamps from MFA TextGrid.

    Returns:
        phones: list of phoneme labels
        intervals: list of (start, end, phone)
    """
    
    tg = textgrid.openTextgrid(tg_path, includeEmptyIntervals=True)
    
    # List available tiers
   # print("Available tiers:", tg.tierNames)
    
    # Usually MFA phoneme tier is named "phones"
    phone_tier = tg.getTier(t)
    
    phones = []
    intervals = []
    
    for start, end, label in phone_tier.entries:
        
        label = label.strip()
        
        # Skip empty intervals
        if label == "":
            continue
        
        # Optionally remove silence
        if remove_silence and label in ["sil", "sp", "spn"]:
            continue
        
        phones.append(label)
        intervals.append((start, end, label))
    
    return phones, intervals

In [28]:
audio_dir = "/vol/corpora/Rhapsodie/wav16k_corrected/"
textgrid_dir = "/vol/corpora/Rhapsodie/TextGrids-fev2013/"
vocab = "vocab_w2vCTC.json"
output="results/rhap_ctc.pkl"
tier = "phone"
device = "cuda"
print(f"Device: {device}")

# ── Processor ─────────────────────────────────────────────────────────
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file           = vocab,
    unk_token            = "[UNK]",
    pad_token            = "[PAD]",
    word_delimiter_token = "",
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size         = 1,
    sampling_rate        = 16000,
    padding_value        = 0.0,
    do_normalize         = True,
    return_attention_mask = True,
)
print("Loading processor...")
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=vocab,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="",
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor, tokenizer=tokenizer
)

ctc_checkpoint = "results/w2vCTC/checkpoint-23430"
# ── Model ─────────────────────────────────────────────────────────────
print(f"Loading CTC architecture from {ctc_checkpoint}...")
base_model = Wav2Vec2ForCTC.from_pretrained(
    ctc_checkpoint,
    ctc_loss_reduction = "mean",
    ctc_zero_infinity  = True,
    pad_token_id       = tokenizer.pad_token_id,
    vocab_size         = len(tokenizer),
)


bin_path = os.path.join(ctc_checkpoint, "pytorch_model.bin")
sft_path = os.path.join(ctc_checkpoint, "model.safetensors")
if os.path.exists(bin_path):
    state_dict = torch.load(bin_path, map_location="cpu")
elif os.path.exists(sft_path):
    from safetensors.torch import load_file
    state_dict = load_file(sft_path)
else:
    raise FileNotFoundError(
        f"No model weights found in {checkpoint}\n"
        f"Expected pytorch_model.bin or model.safetensors"
    )

missing, unexpected = base_model.load_state_dict(state_dict, strict=False)

base_model.eval()
base_model.to(device)
print("Model ready.")

Device: cuda
Loading processor...
Loading CTC architecture from results/w2vCTC/checkpoint-23430...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model ready.


In [29]:
model = base_model

In [30]:
model

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [31]:
audio_files = sorted(
    glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)
)
audio_files[0]

'/vol/corpora/Rhapsodie/wav16k_corrected/Rhap-D0002.wav'

In [32]:
def extract_phoneme_sequence(alignment_list):
    return [item["phoneme"] for item in alignment_list]


In [35]:
def clean_alignment_dict(alignment_list,flag="",is_hyp=False):
    """
    Normalize phonemes and remove empty or deleted ones.
    Keeps timestamps aligned.
    """
    
    cleaned = []
    
    for item in alignment_list:
        phoneme = item["phoneme"]

        phoneme_norm = phoneme, is_hyp=is_hyp
        
        # Remove empty phonemes after normalization
        if phoneme_norm == "" or phoneme_norm is None:
            continue
        
        cleaned.append({
            "phoneme": phoneme_norm,
            "start": item["start"],
            "end": item["end"]
        })
    
    return cleaned

In [40]:
def get_hyp_intervals(clean_hyp):
    """
    Hyp timestamps are always audio-relative (CTC frame * stride).
    No correction needed.
    """
    return list(clean_hyp)

In [54]:
import os
import json
from pathlib import Path
import pandas as pd
import pickle
import unicodedata
from VAD_chunk import *
style = pd.read_csv("/vol/corpora/Rhapsodie/wav_style.csv")
audio_dir = "/vol/corpora/Rhapsodie/wav16k_corrected"
textgrid_dir = Path("/vol/corpora/Rhapsodie/TextGrids-fev2013/")
ref_files = textgrid_dir.glob("*")


alignment_store = {}

for filepath in ref_files:
    f = filepath.stem[:-4] + ".wav"
    audio_path = os.path.join(audio_dir, f)
    
    if "D0001" in audio_path or "D2004" in audio_path:
        continue

    s = list(style[style["file"] == f.split("-")[1].split(".")[0]]["style"])[0]
    textgrid_path = os.path.join(textgrid_dir, f.replace(".wav", "-Pro.TextGrid"))

    pred_phonemes, pred_alignments = get_phoneme_alignments(model, processor, audio_path)
    clean_ref     = read_textgrid(textgrid_path, tier)
    ref_intervals = get_ref_intervals(clean_ref, audio_path)

    #clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
    #clean_ref = clean_alignment_dict(ref_alignments, flag="", is_hyp=False)
    # Fix session offset in ref if needed — no hyp information used
    ref_intervals = get_ref_intervals(ref_intervals, audio_path)
    hyp_intervals = get_hyp_intervals(pred_alignments)

    entry = {
        "file":          f,
        "style":         s,
        "ref_intervals": ref_intervals,
        "hyp_intervals": hyp_intervals,
        "ref_seq":       extract_phoneme_sequence(ref_intervals),
        "hyp_seq":       extract_phoneme_sequence(hyp_intervals),
    }

    alignment_store[f] = entry



  Session offset corrected (end_overshoot): ref_first=0.342s, ref_last=35.106s, audio=34.764s, end_overshoot=+0.342s, offset=0.342s
  Session offset corrected (end_overshoot): ref_first=0.440s, ref_last=277.993s, audio=277.553s, end_overshoot=+0.440s, offset=0.440s
  Session offset corrected (end_overshoot): ref_first=1.138s, ref_last=63.659s, audio=62.521s, end_overshoot=+1.138s, offset=1.138s
  Session offset corrected (end_overshoot): ref_first=0.160s, ref_last=46.557s, audio=46.397s, end_overshoot=+0.160s, offset=0.160s
  Session offset corrected (end_overshoot): ref_first=1.246s, ref_last=308.049s, audio=306.803s, end_overshoot=+1.247s, offset=1.246s
  Session offset corrected (end_overshoot): ref_first=0.125s, ref_last=87.946s, audio=87.821s, end_overshoot=+0.125s, offset=0.125s
  Session offset corrected (end_overshoot): ref_first=1.468s, ref_last=240.883s, audio=239.415s, end_overshoot=+1.468s, offset=1.468s
  Session offset corrected (end_overshoot): ref_first=0.460s, ref_last

LibsndfileError: Error opening '/vol/corpora/Rhapsodie/wav16k_corrected/Rhap-D1003.wav': System error.

In [ ]:
import pickle
from collections import defaultdict

def pkl_to_etf(pkl_path, output_path, use_hyp=True):

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    with open(output_path, "w", encoding="utf-8") as out:

        for filename, content in data.items():

            style = content.get("style", "-")

            key = "hyp_intervals" if use_hyp else "ref_intervals"
            intervals = sorted(content[key], key=lambda x: x["start"])

            if not intervals:
                continue

            # 🔥 récupérer tous les phonèmes présents
            phonemes = sorted(set(seg["phoneme"] for seg in intervals))

            # 🔥 timeline complète (tous segments)
            for target_phoneme in phonemes:

                for seg in intervals:
                    start = seg["start"]
                    end = seg["end"]
                    duration = end - start
                    phoneme = seg["phoneme"]

                    if duration <= 0:
                        continue

                    # 🔥 logique clé
                    decision = "t" if phoneme == target_phoneme else "f"

                    line = (
                        f"{filename} 1 "
                        f"{start:.6f} {duration:.6f} "
                        f"sc - {target_phoneme} - {decision}\n"
                    )

                    out.write(line)

In [ ]:
with open("results/alignment_rhap.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [ ]:
pkl_to_etf("results/alignment_rhap.pkl", "hyp.etf", use_hyp=True)
pkl_to_etf("results/alignment_rhap.pkl", "ref.etf", use_hyp=False)

In [ ]:
from metrics import *
metrics(alignment_store, "metrics_per_file.csv")

In [ ]:
!python trackeval.py --margin=0 --error=event --segmentation=event ref.etf hyp.etf